<h1>Extracting Data from Flight Labs API</h1>

In [1]:
import os
from dotenv import load_dotenv
from utils import extract, transform, load_to_csv, load_to_postgres
from sqlalchemy import create_engine

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
Extraction complete
(100, 23)


<H1>Exploring the Raw Data </h1>

In [2]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     99 non-null     object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   flight_number  100 non-null    object 
 10  flight_icao    100 non-null    object 
 11  flight_iata    100 non-null    object 
 12  dep_icao       100 non-null    object 
 13  dep_iata       100 non-null    object 
 14  arr_icao       100 non-null    object 
 15  arr_iata       100 non-null    object 
 16  airline_icao   100 non-null    object 
 17  airline_iata   100 non-null    object 
 18  aircraft_ic

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,C00757,C-FCUG,CA,49.201945,-123.176469,-7,115.0,39,0,245,...,YEG,CYVR,YVR,ACA,AC,A320,1764811801,landed,adsb,NaN
1,C08853,C-GZQG,CA,45.077987,-76.288946,9164,242.0,599,0,132,...,YUL,CYYZ,YYZ,POE,PD,E295,1764811801,en-route,adsb,NaN
2,A1EC57,N223BZ,US,33.681850,-117.860726,10383,28.0,738,0,817,...,MHT,KMCO,MCO,MXY,MX,BCS3,1764811801,en-route,adsb,NaN
3,ADAB38,N980AV,CO,14.440107,-88.926337,9133,194.8,905,0,325,...,YYZ,MSLP,SAL,TAI,TA,A320,1764811801,en-route,adsb,NaN
4,0C20BD,HP-1851CMP,PA,10.613350,-83.950273,11678,298.7,876,0,721,...,PTY,MMGL,GDL,CMP,CM,B738,1764811801,en-route,adsb,NaN


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.00000,100.000000,100.000000,100.0,1.000000e+02
mean,22.894411,14.262426,9161.07000,179.086000,765.040000,0.0,1.764812e+09
std,21.351682,92.822983,3430.31361,102.413653,192.230602,0.0,0.000000e+00
min,-34.579228,-126.809808,-7.00000,6.100000,39.000000,0.0,1.764812e+09
25%,15.008138,-75.914954,7591.75000,89.350000,688.500000,0.0,1.764812e+09
50%,24.400475,55.247294,10688.00000,174.500000,795.500000,0.0,1.764812e+09
75%,39.974188,103.077248,11388.50000,268.250000,897.000000,0.0,1.764812e+09
max,68.265803,150.145025,12996.00000,353.800000,1127.000000,0.0,1.764812e+09



Number of null values for every column feature



hex               0
reg_number        1
flag              0
lat               0
lng               0
alt               0
dir               0
speed             0
v_speed           0
flight_number     0
flight_icao       0
flight_iata       0
dep_icao          0
dep_iata          0
arr_icao          0
arr_iata          0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk           99
dtype: int64

<h1>Data Cleaning</h1>

<li>Replacing missing values in "squawk" column with "unknown" if the column is pulled during extraction</li>
<li>Replacing missing values in 'alt', 'speed' and v_speed to 0</li>
<li>Renaming columns</li>
<li>Converting "updated" values to datetime</li>


In [3]:
flight_data_clean = transform(flight_data_raw)

display(flight_data_clean.head())

print('\nNumber of null values for every column feature\n')
flight_data_clean.isnull().sum()


transform complete


,hex,reg_number,flag,latitude,longitude,altitude_ft,dir,speed_mph,v_speed_mph,flight_number,...,departure_iata,arrival_icao,arrival_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,C00757,C-FCUG,CA,49.201945,-123.176469,-22.96588,115.0,24.233469,0.0,245,...,YEG,CYVR,YVR,ACA,AC,A320,2025-12-03 20:30:01,landed,adsb,Unknown
1,C08853,C-GZQG,CA,45.077987,-76.288946,30065.61776,242.0,372.201229,0.0,132,...,YUL,CYYZ,YYZ,POE,PD,E295,2025-12-03 20:30:01,en-route,adsb,Unknown
2,A1EC57,N223BZ,US,33.681850,-117.860726,34064.96172,28.0,458.571798,0.0,817,...,MHT,KMCO,MCO,MXY,MX,BCS3,2025-12-03 20:30:01,en-route,adsb,Unknown
3,ADAB38,N980AV,CO,14.440107,-88.926337,29963.91172,194.8,562.340755,0.0,325,...,YYZ,MSLP,SAL,TAI,TA,A320,2025-12-03 20:30:01,en-route,adsb,Unknown
4,0C20BD,HP-1851CMP,PA,10.613350,-83.950273,38313.64952,298.7,544.320996,0.0,721,...,PTY,MMGL,GDL,CMP,CM,B738,2025-12-03 20:30:01,en-route,adsb,Unknown



Number of null values for every column feature



hex               0
reg_number        1
flag              0
latitude          0
longitude         0
altitude_ft       0
dir               0
speed_mph         0
v_speed_mph       0
flight_number     0
flight_icao       0
flight_iata       0
departure_icao    0
departure_iata    0
arrival_icao      0
arrival_iata      0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk            0
dtype: int64

<h1>Loading Cleaned Data to CSVs and Postgres </h1>

In [4]:
# Connecting to local flight data database
dbname=os.getenv('DB_NAME')
user=os.getenv('DB_USER')
password=os.getenv('DB_PASSWORD')
host=os.getenv('DB_HOST')
port=os.getenv('DB_PORT')

conn = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

# Loading raw and clean data
load_to_csv(flight_data_raw, flight_data_clean)
load_to_postgres(flight_data_raw, flight_data_clean, conn)

load complete
